In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os


In [ ]:
try:
    spark.stop()
except:
    pass


In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Practitioner")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())


In [ ]:
silver_base_path = "../../data_lake/silver/silver_practitioner/"
gold_base_path = "../../data_lake/gold/dim_practitioner/"


In [ ]:
df_practitioner = spark.read.format("parquet").load(silver_base_path)


In [ ]:
df_inter = (df_practitioner.select(
    F.conv(F.substring(F.md5(col("practitioner_id")), 1, 15), 16, 10).cast("bigint").alias("practitioner_key"),
    col("practitioner_id"),
    col("npi"),
    col("prefix"),
    col("first_name"),
    col("last_name"),
    col("gender"),
    col("phone"),
    col("email"),
    col("address_line"),
    col("city"),
    col("state"),
    col("postal_code"),
    col("country"),
    col("is_active"),
    col("utilization_encounters"),
    F.current_timestamp().alias("gold_timestamp")
))


In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)


In [ ]:
spark.stop()
